# Phase 6c — The Decisive Ablation

Phase 5b gave "+0.054 PR-AUC, 4/5 folds, **underpowered**". The limit was never
effect size — with 5 folds the two-sided sign test bottoms out at p = 0.0625, so
significance was unreachable by construction.

Two things change here:

1. **2,056 non-redundant clusters** instead of 21.
2. **10 folds instead of 5**, which is the part that actually buys the test its
   power. More data per fold sharpens each estimate; more *folds* is what raises
   the number of independent units the statistic sees.

### Why folds are automatically homology-clean

Every chain in this dataset is the sole representative of its MMseqs2 cluster, so
any two chains share **< 30 % sequence identity by construction**. A plain
`KFold` over chains is therefore already a sequence-clustered split — no grouping
variable required, and no way for a homologue to straddle the boundary.

### On statistics

Fold-level differences (seeds averaged within fold) are the independent units.
Both a **paired t-test** and **Wilcoxon** are reported: Wilcoxon is
distribution-free but floors at p = 0.002 for n = 10, while the t-test can resolve
further when the effect is consistent. Reporting both, and the win count, is
what makes the claim checkable rather than cherry-picked.

**Training hyperparameters differ from Phase 5b** (larger batch, fewer epochs —
50× more data per epoch makes the old schedule wasteful). All three arms receive
*identical* treatment, so the comparison is internally valid; absolute PR-AUC is
not directly comparable to the 41-chain run, and the ablation delta is.

## Cell 1 — Setup

In [ ]:
!pip install -q torch-geometric==2.6.1

In [ ]:
import copy, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold


def locate(fname):
    c = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if root.exists():
            c.extend(root.rglob(fname))
    if not c:
        raise FileNotFoundError(fname)
    return max(c, key=lambda p: p.stat().st_size)


blob = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
TOPO_NAMES = blob["topo_names"]
TOPO_DIM = len(TOPO_NAMES)
N_SEQ = int(dataset[0].x.size(1)) - TOPO_DIM
BURIAL_IDX = [TOPO_NAMES.index(n) for n in blob["burial_names"]]
POS_WEIGHT = float(blob["pos_weight"])

GPU_OK = False
if torch.cuda.is_available():
    try:
        _p = torch.randn(256, 256, device="cuda"); (_p @ _p).sum().item()
        torch.cuda.synchronize(); GPU_OK = True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e:
        print(f"GPU unusable: {str(e).splitlines()[0]}")
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

n_res = sum(int(d.num_nodes) for d in dataset)
n_pos = sum(int(d.y.sum()) for d in dataset)
print(f"device {DEVICE} | chains {len(dataset):,} | residues {n_res:,} "
      f"| positives {n_pos:,} ({100*n_pos/n_res:.2f}%)")
print(f"features: seq {N_SEQ} + topo {TOPO_DIM} | pos_weight {POS_WEIGHT:.2f}")
assert len({d.cluster for d in dataset}) == len(dataset), \
    "chains are not 1:1 with clusters -- KFold would not be homology-clean"
print("verified: 1 chain per cluster -> KFold is automatically a clustered split")

## Cell 2 — Model (unchanged from Phase 3/5)

In [ ]:
class EGCL(nn.Module):
    def __init__(self, h_dim, edge_attr_dim=2, act=nn.SiLU, update_coords=True):
        super().__init__()
        self.update_coords = update_coords
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * h_dim + 1 + edge_attr_dim, h_dim), act(),
            nn.Linear(h_dim, h_dim), act())
        self.node_mlp = nn.Sequential(nn.Linear(2 * h_dim, h_dim), act(),
                                      nn.Linear(h_dim, h_dim))
        self.att = nn.Sequential(nn.Linear(h_dim, 1), nn.Sigmoid())
        if update_coords:
            self.coord_mlp = nn.Sequential(nn.Linear(h_dim, h_dim), act(),
                                           nn.Linear(h_dim, 1, bias=False))
            nn.init.xavier_uniform_(self.coord_mlp[-1].weight, gain=0.001)
        self.norm = nn.LayerNorm(h_dim)

    def forward(self, h, x, edge_index, edge_attr):
        src, dst = edge_index
        rel = x[src] - x[dst]
        d2 = (rel ** 2).sum(-1, keepdim=True)          # only invariant scalars enter MLPs
        m = self.edge_mlp(torch.cat([h[src], h[dst], d2, edge_attr], dim=-1))
        m = m * self.att(m)
        agg = torch.zeros_like(h).index_add_(0, dst, m)
        deg = torch.zeros(h.size(0), 1, device=h.device).index_add_(
            0, dst, torch.ones_like(d2)).clamp(min=1)
        h = self.norm(h + self.node_mlp(torch.cat([h, agg / deg], dim=-1)))
        if self.update_coords:
            w = self.coord_mlp(m) * rel / (d2.sqrt() + 1.0)
            x = x + torch.zeros_like(x).index_add_(0, dst, w) / deg
        return h, x


class TopoFusion(nn.Module):
    def __init__(self, h_dim, topo_dim):
        super().__init__()
        self.proj = nn.Linear(topo_dim, h_dim)
        self.gate = nn.Sequential(nn.Linear(h_dim + topo_dim, h_dim), nn.Sigmoid())

    def forward(self, h, t):
        g = self.gate(torch.cat([h, t], dim=-1))
        self.last_gate = g.detach()
        return h + g * self.proj(t)


class TopologicalEGNN(nn.Module):
    def __init__(self, n_seq, topo_dim, h_dim=128, n_layers=4, edge_attr_dim=2, dropout=0.1):
        super().__init__()
        self.n_seq = n_seq
        self.embed = nn.Sequential(nn.Linear(n_seq + topo_dim, h_dim), nn.SiLU(),
                                   nn.Linear(h_dim, h_dim))
        self.layers = nn.ModuleList([EGCL(h_dim, edge_attr_dim, update_coords=(i < n_layers - 1))
                                     for i in range(n_layers)])
        self.fusions = nn.ModuleList([TopoFusion(h_dim, topo_dim) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(h_dim, h_dim // 2), nn.SiLU(),
                                  nn.Dropout(dropout), nn.Linear(h_dim // 2, 1))

    def forward(self, data):
        h = self.embed(data.x)
        pos, topo = data.pos, data.x[:, self.n_seq:]
        for layer, fuse in zip(self.layers, self.fusions):
            h, pos = layer(h, pos, data.edge_index, data.edge_attr)
            h = fuse(h, topo)
        return self.head(h).squeeze(-1)

## Cell 3 — Equivariance check before anything else

Same assertion as Phase 3, re-run on this dataset. The guarantee depends on the
graphs, so it is re-verified whenever the graphs change.

In [ ]:
@torch.no_grad()
def check_equivariance(n_trials=8):
    model = TopologicalEGNN(N_SEQ, TOPO_DIM).to(DEVICE).eval()
    d = dataset[0].clone().to(DEVICE)
    base = model(d).cpu()
    worst = 0.0
    for s in range(n_trials):
        g = torch.Generator().manual_seed(s)
        Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g))
        Q = (Q * torch.sign(torch.det(Q))).to(DEVICE)
        t = (torch.randn(3, generator=g) * 25.0).to(DEVICE)
        d2 = d.clone(); d2.pos = d.pos @ Q.T + t
        worst = max(worst, float((model(d2).cpu() - base).abs().max()))
    return worst


w = check_equivariance()
print(f"prediction invariance under SE(3): max |dlogit| = {w:.3e}")
assert w < 1e-3, "MODEL IS NOT ROTATION INVARIANT"
print("E(3) guarantee holds on the scaled dataset")

## Cell 4 — Training machinery

In [ ]:
EPOCHS, PATIENCE, BATCH = 40, 8, 16
ORIG_X = [d.x.clone() for d in dataset]


def apply_arm(arm):
    for d, x0 in zip(dataset, ORIG_X):
        t = x0[:, N_SEQ:].clone()
        if arm == "seq-only":
            t.zero_()
        elif arm == "burial-only":
            keep = torch.zeros_like(t); keep[:, BURIAL_IDX] = t[:, BURIAL_IDX]; t = keep
        d.x = torch.cat([x0[:, :N_SEQ], t], dim=1)


def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    ps, ys = [], []
    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(batch)
            loss = crit(logits, batch.y)
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy())
        ys.append(batch.y.detach().cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    return {"roc": roc_auc_score(y, p), "pr": average_precision_score(y, p), "p": p, "y": y}


def best_mcc(p, y):
    return max(matthews_corrcoef(y, (p >= t).astype(int))
               for t in np.linspace(0.05, 0.95, 37) if (p >= t).sum())


def fit(tr_ds, va_ds, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = TopologicalEGNN(N_SEQ, TOPO_DIM).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    tl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
    vl = DataLoader(va_ds, batch_size=BATCH)
    best, state, bad = -1.0, None, 0
    for _ in range(EPOCHS):
        run_epoch(model, tl, opt)
        v = run_epoch(model, vl)
        sched.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(state)
    return model, best

## Cell 5 — 10-fold × 2 seeds × 3 arms

In [ ]:
N_FOLDS, SEEDS = 10, [0, 1]
ARMS = ["seq-only", "burial-only", "full"]

idx_all = np.arange(len(dataset))
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=1337).split(idx_all))

records = []
t_start = time.time()
for arm in ARMS:
    apply_arm(arm)
    for fi, (tr_idx, te_idx) in enumerate(folds):
        for seed in SEEDS:
            rng = np.random.RandomState(500 + seed)
            perm = rng.permutation(tr_idx)
            n_val = max(1, int(0.15 * len(perm)))
            va_ds = [dataset[i] for i in perm[:n_val]]
            tr_ds = [dataset[i] for i in perm[n_val:]]
            te_ds = [dataset[i] for i in te_idx]

            model, val_pr = fit(tr_ds, va_ds, seed)
            te = run_epoch(model, DataLoader(te_ds, batch_size=BATCH))
            records.append({"arm": arm, "fold": fi, "seed": seed, "val_pr": val_pr,
                            "test_pr": te["pr"], "test_roc": te["roc"],
                            "test_mcc": best_mcc(te["p"], te["y"]),
                            "n_test": int(len(te["y"])), "n_pos": int(te["y"].sum())})
        r = records[-1]
        print(f"  {arm:<12} fold {fi:<2} | val PR {r['val_pr']:.3f} | test PR {r['test_pr']:.3f} "
              f"ROC {r['test_roc']:.3f} (n={r['n_test']:,}, pos={r['n_pos']:,}) "
              f"[{(time.time()-t_start)/60:.0f}m]")

for d, x0 in zip(dataset, ORIG_X):
    d.x = x0
print(f"\n{len(records)} runs in {(time.time()-t_start)/60:.1f} min")
Path("/kaggle/working/phase6c_records.json").write_text(json.dumps(records, indent=2, default=float))

## Cell 6 — Statistics

In [ ]:
from scipy.stats import ttest_rel, wilcoxon

def fold_means(arm, metric):
    return np.array([np.mean([r[metric] for r in records
                              if r["arm"] == arm and r["fold"] == f])
                     for f in range(N_FOLDS)])


print(f"{'arm':<14}{'test PR-AUC':>18}{'test ROC-AUC':>18}{'test MCC':>16}")
print("-" * 66)
for arm in ARMS:
    pr, roc, mcc = (fold_means(arm, m) for m in ("test_pr", "test_roc", "test_mcc"))
    print(f"{arm:<14}{pr.mean():>10.4f} +-{pr.std():.4f}"
          f"{roc.mean():>11.4f} +-{roc.std():.4f}{mcc.mean():>9.4f} +-{mcc.std():.4f}")

print("\n" + "=" * 78)
print(f"PAIRED FOLD-LEVEL COMPARISONS (n={N_FOLDS} independent folds, seeds averaged)")
print("=" * 78)
for a, b in (("full", "burial-only"), ("full", "seq-only"), ("burial-only", "seq-only")):
    for metric in ("test_pr", "test_roc"):
        d = fold_means(a, metric) - fold_means(b, metric)
        t_p = ttest_rel(fold_means(a, metric), fold_means(b, metric)).pvalue
        try:
            w_p = wilcoxon(d).pvalue
        except ValueError:
            w_p = float("nan")
        star = "***" if t_p < 0.001 else "**" if t_p < 0.01 else "*" if t_p < 0.05 else "n.s."
        print(f"  {a:<12} - {b:<12} {metric:<9} {d.mean():+.4f} +-{d.std():.4f} | "
              f"{(d > 0).sum():>2}/{N_FOLDS} wins | t p={t_p:.5f} {star} | W p={w_p:.4f}")

d = fold_means("full", "test_pr") - fold_means("burial-only", "test_pr")
print(f"\nHEADLINE: topology beyond burial = {d.mean():+.4f} PR-AUC "
      f"(95% CI {d.mean()-1.96*d.std()/np.sqrt(N_FOLDS):+.4f} to "
      f"{d.mean()+1.96*d.std()/np.sqrt(N_FOLDS):+.4f})")

## Cell 7 — Plot and save

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))
ax[0].boxplot([fold_means(a, "test_pr") for a in ARMS], tick_labels=ARMS)
for i, a in enumerate(ARMS):
    v = fold_means(a, "test_pr")
    ax[0].scatter(np.full(len(v), i + 1) + np.linspace(-.1, .1, len(v)), v, s=16, alpha=.75)
ax[0].set_ylabel("test PR-AUC"); ax[0].set_title(f"{N_FOLDS}-fold CV, 2,056 clusters")

dd = fold_means("full", "test_pr") - fold_means("burial-only", "test_pr")
ax[1].bar(range(N_FOLDS), dd, color=["tab:green" if x > 0 else "tab:red" for x in dd])
ax[1].axhline(0, color="k", lw=1)
ax[1].axhline(dd.mean(), color="tab:blue", ls="--", lw=1.2, label=f"mean {dd.mean():+.4f}")
ax[1].set_xlabel("fold"); ax[1].set_ylabel("PR-AUC difference")
ax[1].set_title("full - burial-only, per fold"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

summary = {a: {m: [float(fold_means(a, m).mean()), float(fold_means(a, m).std())]
               for m in ("test_pr", "test_roc", "test_mcc")} for a in ARMS}
Path("/kaggle/working/phase6c_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

---

## Interpreting this

With 10 independent folds over 2,056 non-redundant clusters, the paired t-test
can now resolve significance — the structural ceiling that made Phase 5b
inconclusive is gone. Whatever this returns is the project's real answer:

- **Significant positive** → persistent homology contributes to binding-site
  prediction beyond burial, sequence, and an equivariant architecture, measured
  on a sequence-clustered split at realistic scale. That is a genuine result.
- **Null** → report it. The Phase 2b feature-level evidence stands on its own,
  and "topological descriptors carry burial-independent signal that a 4-layer
  EGNN does not convert into predictive gain" is a precise, honest, and
  interesting negative finding.

Either way it is now a real measurement rather than a suggestion, which is the
thing that was missing at 41 chains.